**DOCUMENT LOADERS**

To begin implementing Retrieval Augmented Generation (RAG), you'll first need to load the documents that the model will access. These documents can come from a variety of sources, and LangChain supports document loaders for many of them

*PDF Loader*

In [ ]:
#!pip install pypdf
#!pip install langchain-community
#!pip install unstructured

# For LangChain integration
#!pip install -q -U langchain-google-genai

#!pip install -qU "langchain-chroma>=0.1.2"
#!pip install opentelemetry-api==1.38.0 opentelemetry-sdk==1.38.0 opentelemetry-exporter-otlp-proto-grpc==1.38.0 opentelemetry-exporter-otlp-proto-http==1.38.0 opentelemetry-exporter-otlp-proto-common==1.38.0 opentelemetry-proto==1.38.0 --force-reinstall


In [ ]:
#Import the library
from langchain_community.document_loaders import PyPDFLoader, CSVLoader, UnstructuredHTMLLoader

In [ ]:
#Create a document loader for PDF
loader = PyPDFLoader("/content/ACU GS FORM.pdf")

data = loader.load()
print(data[0])

page_content='Simplified Student Visa Framework applicant Genuine Student questionnaire 
Australian Catholic University (ACU) is committed to ensuring that its international applicant population consists of academically able and genuine applicants who are studying in Australia for the purpose of academic, personal and future career development. As part of the Australian Government's Simplified Student Visa Framework (SSVF) arrangements, ACU requires additional information from prospective applicants applying to undertake their studies at ACU. 
APPLICANT DETAILS 
Title: 
Family name: 
Given name: 
Prior name (including name at birth, previous married name or aliases): 
What is your current relationship status (single, married, divorced, widowed, engaged, de facto): 
What is your country of usual residence: 
YOUR STUDIES 
1 Have you previously studied in Australia? 
2 
FINANCIAL CAPACITY 
Ifyou have a question about this checklist, please contact the ACU International Admissions by reply

*CSV Loader*

In [ ]:
#Create a document loader for CSV
loader = CSVLoader(file_path='/content/train.csv')

data = loader.load()
print(data[0])

page_content='datetime: 2011-01-01 00:00:00
season: 1
holiday: 0
workingday: 0
weather: 1
temp: 9.84
atemp: 14.395
humidity: 81
windspeed: 0
casual: 3
registered: 13
count: 16' metadata={'source': '/content/train.csv', 'row': 0}


*HTML Loader*

In [ ]:
#Create a document loader for HTML
loader = UnstructuredHTMLLoader("/content/yahoo.html")

data = loader.load()
print(data[0])

#Print the first elements metadata
print(data[0].metadata)

page_content='2of 15



Dire Iran war revelation from the White House no Australian wants to hear



CNN



'50 CUT OUT': Bindi Irwin's sad three-year update



Yahoo Lifestyle



Ally sends more troops to Middle East just hours after Trump's warning



The Independent



Shoppers strip shelves as 'super' fruit imported into Australia for first time ever



Yahoo Lifestyle



Tasmanian tiger extinction in doubt after rare cave discovery



Yahoo News Australia



Common recycling mistake costing Aussie households



Yahoo News Australia



'WE WON'T BE THERE': Trump's brutal message to allies



Associated Press



'LITERALLY EVERYONE': Aussie tech worker on Silicon Valley's 'wild' new trend



Yahoo Finance AU



Plea to Aussies as fuel misconception wreaks havoc ahead of Easter



Yahoo News Australia



Major perks to disappear as RBA change becomes official: 'Much sooner'



Yahoo Finance AU



Eyebrow-raising Harry and Meghan claim shocks Channel 9 viewers



Yahoo Lifestyle



Wo

**SPLITTING EXTERNAL DATA FOR RETRIVAL**

A key process in implementing Retrieval Augmented Generation (RAG) is splitting documents into chunks for storage in a vector database.

There are several splitting strategies available in LangChain, some with more complex routines than others

*Character splitting*

In [ ]:
#Import the character splitter
from langchain_text_splitters import CharacterTextSplitter

quote = "Words are flowing out like endless rain into a paper cup,\nthey slither while they pass,\nthey slip away across the universe"
chunk_size=24
chunk_overlap=10

#Create an instance of the splitter class
splitter = CharacterTextSplitter(
    separator = "\n",
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

#Split the string and print the chunks
chunks = splitter.split_text(quote)
print(chunks)
print([len(chunks) for chunk in chunks])

['Words are flowing out like endless rain into a paper cup,', 'they slither while they pass,', 'they slip away across the universe']
[3, 3, 3]


*Recursive character splitting*

In [ ]:
#Import the recursive character splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

quote = "Words are flowing out like endless rain into a paper cup,\nthey slither while they pass,\nthey slip away across the universe"
chunk_size=24
chunk_overlap=10

#Create an instance of the splitter class
splitter = RecursiveCharacterTextSplitter(
    separators=["\n"," ",""],
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap)

#Split the string and print the chunks
chunks = splitter.split_text(quote)
print(chunks)
print([len(chunks) for chunk in chunks])

['Words are flowing out', 'out like endless rain', 'rain into a paper cup,', 'they slither while they', 'they pass,', 'they slip away across', 'across the universe']
[7, 7, 7, 7, 7, 7, 7]


*Splitting HTML*

In [ ]:
#Load the HTML document into memory
from langchain_community.document_loaders import UnstructuredHTMLLoader
loader = UnstructuredHTMLLoader("/content/yahoo.html")
data = loader.load()

chunk_size = 300
chunk_overlap = 100

#Create an instance for the splitter class
splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators="."
)

#Split the string and print the chunks
docs = splitter.split_documents(data)
print(docs)

[Document(metadata={'source': '/content/yahoo.html'}, page_content='2of 15\n\n\n\nDire Iran war revelation from the White House no Australian wants to hear\n\n\n\nCNN\n\n\n\n\'50 CUT OUT\': Bindi Irwin\'s sad three-year update\n\n\n\nYahoo Lifestyle\n\n\n\nAlly sends more troops to Middle East just hours after Trump\'s warning\n\n\n\nThe Independent\n\n\n\nShoppers strip shelves as \'super\' fruit imported into Australia for first time ever\n\n\n\nYahoo Lifestyle\n\n\n\nTasmanian tiger extinction in doubt after rare cave discovery\n\n\n\nYahoo News Australia\n\n\n\nCommon recycling mistake costing Aussie households\n\n\n\nYahoo News Australia\n\n\n\n\'WE WON\'T BE THERE\': Trump\'s brutal message to allies\n\n\n\nAssociated Press\n\n\n\n\'LITERALLY EVERYONE\': Aussie tech worker on Silicon Valley\'s \'wild\' new trend\n\n\n\nYahoo Finance AU\n\n\n\nPlea to Aussies as fuel misconception wreaks havoc ahead of Easter\n\n\n\nYahoo News Australia\n\n\n\nMajor perks to disappear as RBA chang

**RAG STORAGE AND RETRIEVAL USING VECTOR DATABASE**

We'll build a full RAG workflow to have a conversation with a PDF document containing the paper, RAG VS Fine-Tuning: Pipelines, Tradeoffs, and a Case Study on Agriculture by Balaguer et al. (2024). This works by splitting the documents into chunks, storing them in a vector database, defining a prompt to connect the retrieved documents and user input, and building a retrieval chain for the LLM to access this external data.

In [ ]:
#Import the library
from langchain_google_genai import ChatGoogleGenerativeAI

from google.colab import userdata
API_KEY_NEW = userdata.get('RAG_AI')
genai.configure(api_key=API_KEY_NEW)

# List all available embedding models
print("Available Embedding Models:")
for m in genai.list_models():
    if "embedContent" in m.supported_generation_methods:
        print(m.name)

Available Embedding Models:
models/gemini-embedding-001
models/gemini-embedding-2-preview


In [ ]:
#Import the library
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import google.generativeai as genai
import os


loader = PyPDFLoader("/content/2401.08406v3.pdf")
data = loader.load()

#Split the documents using the RecursiveCharcterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

docs = splitter.split_documents(data)

#Embed the documents in a persisitent Chroma Vector Database
embedding_function =HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    docs,
    embedding=embedding_function,
    persist_directory="./chroma_db"
)

#Configure the Vector Store as a Retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

/tmp/ipykernel_17790/1531134244.py:22: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function =HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
#Import library
from langchain_core.prompts import ChatPromptTemplate

#Add placeholders to the message string
message = """
Answer the question based only on the following context:
Context:
{context}

Question:
{question}

Answer:
"""

#Create a chat prompt template from the message string
prompt_template = ChatPromptTemplate.from_messages(["human",message])

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI

#Create RAG Chain
vectorstore = Chroma.from_documents(
    docs,
    embedding = embedding_function,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs = {"k":3})

# Initialize the LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=API_KEY_NEW)

#Create a chain to link retirever, prompt template and llm
rag_chain = ({"context":retriever,"question":RunnablePassthrough()})| prompt_template |llm

In [ ]:
#Invoke the chain
response = rag_chain.invoke("Which popular LLMs were considered in the paper?")
print(response.content)

Based on the provided context, the document states that "LLMs themselves are judges that display high agreement with humans" and references works by Chiang and Lee (2023) and Dubois et al. (2023). However, it does not name any specific popular LLMs that were considered in *this* paper.


We have successfully set up a RAG workflow to allow you to talk with a PDF document

In [ ]:
!pip freeze > requirements.txt
